In [68]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

## ml items
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier,GradientBoostingClassifier
from xgboost import XGBClassifier

## handling imbalance data
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import ADASYN
from imblearn.pipeline import Pipeline

In [4]:
os.getcwd()

'c:\\Users\\AnuragS\\OneDrive\\ml_projects\\telecom_churn_v2'

In [84]:
df2 = pd.read_csv('customer_churn.csv')
df2.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [87]:
xcol = ['gender','SeniorCitizen','Partner','Dependents','tenure','PhoneService','MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies','Contract','PaperlessBilling','PaymentMethod']

for col in xcol:
    print(f'{col}')
    print(df2[col].unique())

gender
<ArrowStringArray>
['Female', 'Male']
Length: 2, dtype: str
SeniorCitizen
[0 1]
Partner
<ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str
Dependents
<ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
tenure
[ 1 34  2 45  8 22 10 28 62 13 16 58 49 25 69 52 71 21 12 30 47 72 17 27
  5 46 11 70 63 43 15 60 18 66  9  3 31 50 64 56  7 42 35 48 29 65 38 68
 32 55 37 36 41  6  4 33 67 23 57 61 14 20 53 40 59 24 44 19 54 51 26  0
 39]
PhoneService
<ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
MultipleLines
<ArrowStringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str
InternetService
<ArrowStringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str
OnlineSecurity
<ArrowStringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
OnlineBackup
<ArrowStringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str
DeviceProtection
<ArrowStringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
TechSupport
<ArrowStringArray>


In [5]:
df = pd.read_csv('customer_churn_cleaned.csv')
df.head()

,gender,SeniorCitizen,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_bucket
0,Female,0,Yes,No,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,0-3 Months
1,Male,0,No,No,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0,25-48 Months
2,Male,0,No,No,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,0-3 Months
3,Male,0,No,No,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0,25-48 Months
4,Female,0,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,0-3 Months


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7032 non-null   str    
 1   SeniorCitizen     7032 non-null   int64  
 2   Partner           7032 non-null   str    
 3   Dependents        7032 non-null   str    
 4   PhoneService      7032 non-null   str    
 5   MultipleLines     7032 non-null   str    
 6   InternetService   7032 non-null   str    
 7   OnlineSecurity    7032 non-null   str    
 8   OnlineBackup      7032 non-null   str    
 9   DeviceProtection  7032 non-null   str    
 10  TechSupport       7032 non-null   str    
 11  StreamingTV       7032 non-null   str    
 12  StreamingMovies   7032 non-null   str    
 13  Contract          7032 non-null   str    
 14  PaperlessBilling  7032 non-null   str    
 15  PaymentMethod     7032 non-null   str    
 16  MonthlyCharges    7032 non-null   float64
 17  TotalC

In [7]:
df.Churn.value_counts(normalize=True)*100

Churn
0    73.421502
1    26.578498
Name: proportion, dtype: float64

## Churn Rate = 26%

In [8]:
numeric_cols = list(df.select_dtypes(exclude='str').columns)
numeric_cols = [col for col in numeric_cols if col not in ['SeniorCitizen','Churn']]

categorical_col = list(df.select_dtypes(include='str').columns)

print(f'Numerical columns are: {numeric_cols}')
print(f'Categorical columns are: {categorical_col}')

Numerical columns are: ['MonthlyCharges', 'TotalCharges']
Categorical columns are: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure_bucket']


In [9]:
X = df.drop(['Churn'],axis=1)
y = df.Churn

print('-'*35)
print('-'*35)
print(f'Shape of X: {X.shape}')
print(f'Shape of y: {y.shape}')
print('-'*35)
print('-'*35)

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42)

print('-'*35)
print('-'*35)
print(f'Train data size: {len(X_train)}')
print(f'Test data size: {len(X_test)}')
print('-'*35)
print('-'*35)

-----------------------------------
-----------------------------------
Shape of X: (7032, 19)
Shape of y: (7032,)
-----------------------------------
-----------------------------------
-----------------------------------
-----------------------------------
Train data size: 5274
Test data size: 1758
-----------------------------------
-----------------------------------


In [10]:
pre_proc = ColumnTransformer(
    transformers=[
        ('numerical',StandardScaler(),numeric_cols),
        ('categorical',OneHotEncoder(handle_unknown='ignore'),categorical_col)
    ]
)

X_train_scaled = pre_proc.fit_transform(X_train)
X_test_scaled = pre_proc.transform(X_test)

In [51]:
classification_report_record = []

In [29]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42,class_weight='balanced'),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False,random_state=42)
}

In [52]:
for model_name,model_structure in models.items():
    # print('-'*35)
    # print('-'*35)
    # print(f'\n Model: {model_name}')

    model_structure.fit(X_train_scaled,y_train)
    y_pred = model_structure.predict(X_test_scaled)

    reportx = classification_report(y_pred,y_test,output_dict=True)

    row = {
    'model_name': model_name,
    'technique_to_imbalance_handle': 'before handling imbalance',
    
    # Class 0
    'precision_0': reportx['0']['precision'],
    'recall_0': reportx['0']['recall'],
    'f1_score_0': reportx['0']['f1-score'],
    
    # Class 1
    'precision_1': reportx['1']['precision'],
    'recall_1': reportx['1']['recall'],
    'f1_score_1': reportx['1']['f1-score'],
    
    # Overall
    'accuracy': accuracy_score(y_test, y_pred)
     }
    
    classification_report_record.append(row)

    df_classification_report_record = pd.DataFrame(classification_report_record)

c:\Users\AnuragS\OneDrive\ml_projects\telecom_churn_v2\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [03:01:42] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [53]:
df_classification_report_record

,model_name,technique_to_imbalance_handle,precision_0,recall_0,f1_score_0,precision_1,recall_1,f1_score_1,accuracy
0,Decision Tree,before handling imbalance,0.810000,0.822656,0.816279,0.504367,0.483264,0.493590,0.730375
1,Random Forest,before handling imbalance,0.886923,0.821225,0.852811,0.451965,0.584746,0.509852,0.773606
2,AdaBoost,before handling imbalance,0.899231,0.829666,0.863049,0.475983,0.624642,0.540273,0.788965
3,Gradient Boosting,before handling imbalance,0.895385,0.834409,0.863822,0.495633,0.625344,0.552984,0.791240
4,XGBoost,before handling imbalance,0.870769,0.828091,0.848894,0.486900,0.570332,0.525324,0.770762


From this analysis we get that these models are poorly performing to detect the churners. Also model is giving 76%. This is happening because of imbalance dataset

## Handling Imbalance Data

Techniques used
1. Random Oversampler (Duplicates minority samples)
2. SMOTE (Create synthetic data instead of copying) - variants(SMOTENN,SMOTETomek)
3. RandomUnderSampler (Removes majority samples) - Risk of information loss

## SMOTEENN (Upsampling + ENN)

In [31]:
smenn = SMOTEENN()

X_train_scaled_smoteenn,y_train_smoteenn=smenn.fit_resample(X_train_scaled,y_train)

In [ ]:
for model_name,model_structure in models.items():
    # print('-'*35)
    # print('-'*35)
    # print(f'\n Model: {model_name}')

    model_structure.fit(X_train_scaled_smoteenn,y_train_smoteenn)
    y_pred_smoteenn = model_structure.predict(X_test_scaled)

    reportx = classification_report(y_pred_smoteenn,y_test,output_dict=True)

    row = {
    'model_name': model_name,
    'technique_to_imbalance_handle': 'SMOTEENN',
    
    # Class 0
    'precision_0': reportx['0']['precision'],
    'recall_0': reportx['0']['recall'],
    'f1_score_0': reportx['0']['f1-score'],
    
    # Class 1
    'precision_1': reportx['1']['precision'],
    'recall_1': reportx['1']['recall'],
    'f1_score_1': reportx['1']['f1-score'],
    
    # Overall
    'accuracy': accuracy_score(y_test, y_pred_smoteenn)
     }
    
    classification_report_record.append(row)

    df_classification_report_record = pd.DataFrame(classification_report_record)

-----------------------------------
-----------------------------------

 Model: Decision Tree
-----------------------------------
-----------------------------------

 Model: Random Forest
-----------------------------------
-----------------------------------

 Model: AdaBoost
-----------------------------------
-----------------------------------

 Model: Gradient Boosting
-----------------------------------
-----------------------------------

 Model: XGBoost


c:\Users\AnuragS\OneDrive\ml_projects\telecom_churn_v2\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [03:04:19] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [55]:
df_classification_report_record

,model_name,technique_to_imbalance_handle,precision_0,recall_0,f1_score_0,precision_1,recall_1,f1_score_1,accuracy
0,Decision Tree,before handling imbalance,0.810000,0.822656,0.816279,0.504367,0.483264,0.493590,0.730375
1,Random Forest,before handling imbalance,0.886923,0.821225,0.852811,0.451965,0.584746,0.509852,0.773606
2,AdaBoost,before handling imbalance,0.899231,0.829666,0.863049,0.475983,0.624642,0.540273,0.788965
3,Gradient Boosting,before handling imbalance,0.895385,0.834409,0.863822,0.495633,0.625344,0.552984,0.791240
4,XGBoost,before handling imbalance,0.870769,0.828091,0.848894,0.486900,0.570332,0.525324,0.770762
5,Decision Tree,SMOTEENN,0.688462,0.894106,0.777923,0.768559,0.464993,0.579424,0.709329
6,Random Forest,SMOTEENN,0.699231,0.905378,0.789062,0.792576,0.481432,0.599010,0.723549
7,AdaBoost,SMOTEENN,0.650769,0.930693,0.765957,0.862445,0.465253,0.604438,0.705916
8,Gradient Boosting,SMOTEENN,0.677692,0.918665,0.779991,0.829694,0.475594,0.604614,0.717292
9,XGBoost,SMOTEENN,0.698462,0.899901,0.786488,0.779476,0.476636,0.591549,0.719568


In [33]:
sm = SMOTE()

X_train_scaled_smote,y_train_smote=sm.fit_resample(X_train_scaled,y_train)

In [56]:
for model_name,model_structure in models.items():
    # print('-'*35)
    # print('-'*35)
    # print(f'\n Model: {model_name}')

    model_structure.fit(X_train_scaled_smote,y_train_smote)
    y_pred_smote = model_structure.predict(X_test_scaled)

    reportx = classification_report(y_pred_smote,y_test,output_dict=True)

    row = {
    'model_name': model_name,
    'technique_to_imbalance_handle': 'SMOTE',
    
    # Class 0
    'precision_0': reportx['0']['precision'],
    'recall_0': reportx['0']['recall'],
    'f1_score_0': reportx['0']['f1-score'],
    
    # Class 1
    'precision_1': reportx['1']['precision'],
    'recall_1': reportx['1']['recall'],
    'f1_score_1': reportx['1']['f1-score'],
    
    # Overall
    'accuracy': accuracy_score(y_test, y_pred_smote)
     }
    
    classification_report_record.append(row)

    df_classification_report_record = pd.DataFrame(classification_report_record)

c:\Users\AnuragS\OneDrive\ml_projects\telecom_churn_v2\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [03:05:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [36]:
ads = ADASYN()

X_train_scaled_ads,y_train_ads=ads.fit_resample(X_train_scaled,y_train)

In [57]:
for model_name,model_structure in models.items():
    # print('-'*35)
    # print('-'*35)
    # print(f'\n Model: {model_name}')

    model_structure.fit(X_train_scaled_ads,y_train_ads)
    y_pred_ads = model_structure.predict(X_test_scaled)

    reportx = classification_report(y_pred_ads,y_test,output_dict=True)

    row = {
    'model_name': model_name,
    'technique_to_imbalance_handle': 'ADASYN',
    
    # Class 0
    'precision_0': reportx['0']['precision'],
    'recall_0': reportx['0']['recall'],
    'f1_score_0': reportx['0']['f1-score'],
    
    # Class 1
    'precision_1': reportx['1']['precision'],
    'recall_1': reportx['1']['recall'],
    'f1_score_1': reportx['1']['f1-score'],
    
    # Overall
    'accuracy': accuracy_score(y_test, y_pred_ads)
     }
    
    classification_report_record.append(row)

    df_classification_report_record = pd.DataFrame(classification_report_record)

c:\Users\AnuragS\OneDrive\ml_projects\telecom_churn_v2\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [03:06:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [60]:
df_classification_report_record.to_csv('model_wise_classification_report.csv',index=False)

📊 Supporting Justification from Your Results
AdaBoost + ADASYN showed:
✅ Best F1-score for class 1 (≈ highest among all combinations)
✅ Better balance between precision and recall
Other models:
Either had higher accuracy but poor minority detection
Or lower F1-score for churn class
🧠 Technical Reasoning

“ADASYN helped by generating more synthetic samples in difficult regions of the minority class, while AdaBoost further improved performance by focusing on misclassified samples during training. This combination enhanced the model’s ability to correctly identify churners.”

💼 Business Justification (Telecom Churn)

“In a telecom churn problem, identifying potential churners is more critical than maximizing overall accuracy. AdaBoost with ADASYN provided the best trade-off by improving detection of churners while maintaining reasonable precision, making it the most suitable model for this use case.”

⚖️ Trade-off Acknowledgment (Very Important)

“Although some models had higher overall accuracy, they underperformed in detecting churners. Since missing churners has a higher business cost, I prioritized F1-score and recall for the minority class.”

🏁 One-Line Power Answer

“I selected AdaBoost with ADASYN because it delivered the highest F1-score for the churn class, indicating the best balance between precision and recall for detecting customers likely to churn.”

In [76]:
base_estimator = DecisionTreeClassifier(max_depth=1)  # default stump

pipeline = Pipeline([
    ('preprocessing', pre_proc),
    ('adasyn', ADASYN()),
    ('model', AdaBoostClassifier())
])

param_grid = {
    # ADASYN
    'adasyn__n_neighbors': [3, 5, 7],
    'adasyn__sampling_strategy': [0.6, 0.8, 1.0],
    
    # AdaBoost
    'model__n_estimators': [100, 200, 500],
    'model__learning_rate': [0.01, 0.05, 0.1, 1],
    
    # Base estimator depth
    'model__estimator': [
        DecisionTreeClassifier(max_depth=1),
        DecisionTreeClassifier(max_depth=2),
        DecisionTreeClassifier(max_depth=3)
    ]
}

search = RandomizedSearchCV(
    pipeline,
    param_grid,
    scoring='f1',   # IMPORTANT
    cv=5,
    n_iter=20,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

best_model = search.best_estimator_

print(search.best_params_)

y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred))

{'model__n_estimators': 500, 'model__learning_rate': 0.05, 'model__estimator': DecisionTreeClassifier(max_depth=3), 'adasyn__sampling_strategy': 0.6, 'adasyn__n_neighbors': 5}
              precision    recall  f1-score   support

           0       0.86      0.84      0.85      1300
           1       0.57      0.61      0.59       458

    accuracy                           0.78      1758
   macro avg       0.71      0.72      0.72      1758
weighted avg       0.78      0.78      0.78      1758



In [77]:
y_prob = best_model.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.2, 0.6, 0.05)

results1 = []

for t in thresholds:
    y_pred = (y_prob > t).astype(int)
    
    results1.append({
        'threshold': round(t, 2),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred)
    })

# Convert to DataFrame
threshold_df_1 = pd.DataFrame(results1)

# Find best threshold
best_row = threshold_df_1.loc[threshold_df_1['f1_score'].idxmax()]

print("\n📊 Threshold Tuning Results:\n")
print(threshold_df_1)

print("\n🏆 Best Threshold Based on F1-score:\n")
print(best_row)


📊 Threshold Tuning Results:

   threshold  precision    recall  f1_score
0       0.20   0.317672  0.989083  0.480892
1       0.25   0.350548  0.978166  0.516129
2       0.30   0.385980  0.949782  0.548896
3       0.35   0.428571  0.910480  0.582809
4       0.40   0.472661  0.849345  0.607338
5       0.45   0.510981  0.762009  0.611744
6       0.50   0.565041  0.606987  0.585263
7       0.55   0.638806  0.467249  0.539723

🏆 Best Threshold Based on F1-score:

threshold    0.450000
precision    0.510981
recall       0.762009
f1_score     0.611744
Name: 5, dtype: float64


In [78]:
y_prob = best_model.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.3, 0.7, 0.05)
#thresholds = np.arange(0.2, 0.6, 0.05)

results2 = []

for t in thresholds:
    y_pred = (y_prob > t).astype(int)
    
    results2.append({
        'threshold': round(t, 2),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred)
    })

# Convert to DataFrame
threshold_df_2 = pd.DataFrame(results2)

# Find best threshold
best_row = threshold_df_2.loc[threshold_df_2['f1_score'].idxmax()]

print("\n📊 Threshold Tuning Results:\n")
print(threshold_df_2)

print("\n🏆 Best Threshold Based on F1-score:\n")
print(best_row)


📊 Threshold Tuning Results:

   threshold  precision    recall  f1_score
0       0.30   0.385980  0.949782  0.548896
1       0.35   0.428571  0.910480  0.582809
2       0.40   0.472661  0.849345  0.607338
3       0.45   0.510981  0.762009  0.611744
4       0.50   0.565041  0.606987  0.585263
5       0.55   0.638806  0.467249  0.539723
6       0.60   0.746988  0.270742  0.397436
7       0.65   0.923077  0.078603  0.144869

🏆 Best Threshold Based on F1-score:

threshold    0.450000
precision    0.510981
recall       0.762009
f1_score     0.611744
Name: 3, dtype: float64


In [ ]:
threshold = 0.45
y_pred_custom = (y_prob > threshold).astype(int)

print(classification_report(y_test, y_pred_custom))

              precision    recall  f1-score   support

           0       0.90      0.74      0.81      1300
           1       0.51      0.76      0.61       458

    accuracy                           0.75      1758
   macro avg       0.70      0.75      0.71      1758
weighted avg       0.80      0.75      0.76      1758



In [81]:
import joblib

best_model = search.best_estimator_
threshold = best_row['threshold']  # from your tuning

joblib.dump(best_model, "model.pkl")
joblib.dump(threshold, "threshold.pkl")

['threshold.pkl']